# LeetCode #93 — Restore IP Addresses: backtracking chia chuỗi thành 4 phần

**Mức độ:** Medium · **Chủ đề:** String, Backtracking

Skeleton trên LeetCode:

```python
class Solution:
    def restoreIpAddresses(self, s: str) -> List[str]:
```

Mục tiêu của notebook:

1. Viết hàm kiểm tra **một phần** IP hợp lệ (`0..255`, không có số 0 ở đầu).
2. Backtracking: mỗi tầng chọn độ dài `1`, `2` hoặc `3` cho phần tiếp theo.
3. **Cắt tỉa** theo số ký tự còn lại (mỗi phần còn lại cần 1 tới 3 ký tự).
4. So sánh với cách **ba vòng lặp lồng nhau** đặt 3 dấu chấm.

# 1. Bài toán là gì?

Một **địa chỉ IP hợp lệ** gồm đúng **bốn số nguyên** cách nhau bởi dấu chấm. Mỗi số nằm trong `[0, 255]` và **không có số 0 ở đầu** (trừ chính số `0`).

- Hợp lệ: `"0.1.2.201"`, `"192.168.1.1"`
- Không hợp lệ: `"0.011.255.245"` (`011` có số 0 đầu), `"192.168.1.312"` (`312 > 255`), `"192.168@1.1"`

Cho chuỗi `s` chỉ gồm chữ số, trả về **mọi** IP hợp lệ có thể tạo ra bằng cách **chèn dấu chấm** vào `s`. **Không được** đổi thứ tự hay xóa chữ số. Thứ tự kết quả tùy ý.

**Constraints (theo đề):**

- `1 <= s.length <= 20`
- `s` chỉ gồm chữ số.

---

## Ví dụ

```text
"25525511135"  ->  ["255.255.11.135", "255.255.111.35"]
"0000"         ->  ["0.0.0.0"]
"101023"       ->  ["1.0.10.23", "1.0.102.3", "10.1.0.23", "10.10.2.3", "101.0.2.3"]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quay lui (backtracking)
2. Chuỗi (string) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [1]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [2]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [3]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [4]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [5]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.2 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [6]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [7]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [8]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [9]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.3 Áp dụng vào bài này

Chia chuỗi thành **4 phần** bằng **quay lui**: mỗi tầng cắt 1–3 ký tự làm một phần; cắt tỉa khi phần đó không hợp lệ (có số 0 đầu, lớn hơn 255) hoặc số ký tự còn lại không đủ/quá nhiều cho các phần còn lại.

# 3. Nền tảng: kiểm tra một phần

Một phần `p` hợp lệ khi:

- `1 <= len(p) <= 3`
- `p == "0"` hoặc `p[0] != "0"` (không có số 0 đầu)
- `int(p) <= 255`

In [10]:
from typing import List
import itertools, random

def valid_part(p: str) -> bool:
    if not 1 <= len(p) <= 3:
        return False
    if p[0] == "0" and len(p) > 1:
        return False
    return int(p) <= 255

for p in ["0", "00", "01", "10", "255", "256", "100", ""]:
    print(f"{p!r:>6} -> {valid_part(p)}")

   '0' -> True
  '00' -> False
  '01' -> False
  '10' -> True
 '255' -> True
 '256' -> False
 '100' -> True
    '' -> False


# 4. Cách 1: Ba vòng lặp đặt 3 dấu chấm

Có đúng 3 dấu chấm, chia `s` thành 4 phần. Mỗi phần dài 1–3 → thử mọi độ dài `a, b, c` cho 3 phần đầu; phần thứ tư là phần còn lại.

In [11]:
def restore_loops(s: str) -> List[str]:
    n = len(s)
    result = []
    for a in range(1, 4):
        for b in range(1, 4):
            for c in range(1, 4):
                d = n - a - b - c
                if not 1 <= d <= 3:
                    continue
                parts = [s[:a], s[a:a + b], s[a + b:a + b + c], s[a + b + c:]]
                if all(valid_part(p) for p in parts):
                    result.append(".".join(parts))
    return result

print(restore_loops("25525511135"))
print(restore_loops("0000"))

['255.255.11.135', '255.255.111.35']
['0.0.0.0']


Chỉ `3 · 3 · 3 = 27` tổ hợp — rất nhanh. Cách này gọn vì số phần **cố định** (4). Nhưng backtracking tổng quát hơn (Bài 4, Bài 5) và là thứ phỏng vấn muốn thấy.

# 5. Cách 2: Backtracking

`backtrack(start, parts)`: đã chọn `parts` (danh sách các phần), đang ở vị trí `start` của `s`.

- Nếu đã có 4 phần: hợp lệ nếu `start == len(s)` (dùng hết chuỗi).
- Ngược lại, thử độ dài `L = 1, 2, 3` cho phần tiếp theo, nếu `s[start:start+L]` hợp lệ thì đi tiếp.

**Cắt tỉa:** còn `remaining = len(s) − start` ký tự cho `k = 4 − len(parts)` phần → cần `k <= remaining <= 3k`. Nếu không thỏa thì dừng ngay.

In [12]:
def restore_backtrack(s: str) -> List[str]:
    result, parts = [], []
    def backtrack(start):
        k = 4 - len(parts)
        remaining = len(s) - start
        if remaining < k or remaining > 3 * k:
            return
        if k == 0:
            result.append(".".join(parts))
            return
        for L in range(1, 4):
            p = s[start:start + L]
            if len(p) < L:
                break
            if not valid_part(p):
                continue
            parts.append(p)
            backtrack(start + L)
            parts.pop()
    backtrack(0)
    return result

print(restore_backtrack("101023"))

['1.0.10.23', '1.0.102.3', '10.1.0.23', '10.10.2.3', '101.0.2.3']


Khi `k == 0` thì `remaining` phải bằng 0 (vì `0 <= remaining <= 0`) — điều kiện cắt tỉa đã kiểm tra luôn việc "dùng hết chuỗi".

# 6. Mô phỏng

In [13]:
def trace_restore(s: str) -> List[str]:
    result, parts = [], []
    def backtrack(start, depth):
        ind = "  " * depth
        k = 4 - len(parts)
        remaining = len(s) - start
        if remaining < k or remaining > 3 * k:
            print(f"{ind}{parts}: còn {remaining} ký tự cho {k} phần -> cắt tỉa")
            return
        if k == 0:
            print(f"{ind}{parts} -> {'.'.join(parts)} ✓")
            result.append(".".join(parts))
            return
        for L in range(1, 4):
            p = s[start:start + L]
            if len(p) < L:
                break
            if not valid_part(p):
                print(f"{ind}{parts} + {p!r}: phần không hợp lệ")
                continue
            parts.append(p)
            backtrack(start + L, depth + 1)
            parts.pop()
    backtrack(0, 0)
    return result

_ = trace_restore("0000")

        ['0', '0', '0', '0'] -> 0.0.0.0 ✓
    ['0', '0'] + '00': phần không hợp lệ
  ['0'] + '00': phần không hợp lệ
  ['0'] + '000': phần không hợp lệ
[] + '00': phần không hợp lệ
[] + '000': phần không hợp lệ


`"00"` và `"000"` bị loại ngay (số 0 ở đầu), nên chỉ còn đúng một nhánh `0.0.0.0`.

# 7. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def restoreIpAddresses(self, s: str) -> List[str]:
        result, parts = [], []

        def valid(p: str) -> bool:
            return (p == "0" or p[0] != "0") and int(p) <= 255

        def backtrack(start: int) -> None:
            k = 4 - len(parts)
            remaining = len(s) - start
            if remaining < k or remaining > 3 * k:
                return
            if k == 0:
                result.append(".".join(parts))
                return
            for L in range(1, min(3, remaining) + 1):
                p = s[start:start + L]
                if valid(p):
                    parts.append(p)
                    backtrack(start + L)
                    parts.pop()

        backtrack(0)
        return result

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def brute_restore(s: str) -> List[str]:
    out = []
    for cuts in itertools.combinations(range(1, len(s)), 3):
        a, b, c = cuts
        parts = [s[:a], s[a:b], s[b:c], s[c:]]
        if all(valid_part(p) for p in parts):
            out.append(".".join(parts))
    return out


sol = Solution()
check("Ví dụ 1", sorted(sol.restoreIpAddresses("25525511135")), ["255.255.11.135", "255.255.111.35"])
check("Ví dụ 2", sol.restoreIpAddresses("0000"), ["0.0.0.0"])
check("Ví dụ 3", sorted(sol.restoreIpAddresses("101023")), ["1.0.10.23", "1.0.102.3", "10.1.0.23", "10.10.2.3", "101.0.2.3"])
check("Quá ngắn", sol.restoreIpAddresses("123"), [])
check("Quá dài", sol.restoreIpAddresses("1" * 13), [])
check("Đúng 12 chữ số", sol.restoreIpAddresses("255255255255"), ["255.255.255.255"])
check("256 không được", sol.restoreIpAddresses("256256256256"), [])

random.seed(93)
for _ in range(3000):
    s = "".join(random.choice("0012259") for _ in range(random.randint(1, 14)))
    expected = sorted(brute_restore(s))
    assert sorted(sol.restoreIpAddresses(s)) == expected == sorted(restore_loops(s)) == sorted(restore_backtrack(s)), s
print("3000 chuỗi ngẫu nhiên: khớp brute force (thử mọi vị trí 3 dấu chấm)")

Ví dụ 1            PASS  ['255.255.11.135', '255.255.111.35']
Ví dụ 2            PASS  ['0.0.0.0']
Ví dụ 3            PASS  ['1.0.10.23', '1.0.102.3', '10.1.0.23', '10.10.2.3', '101.0.2.3']
Quá ngắn           PASS  []
Quá dài            PASS  []
Đúng 12 chữ số     PASS  ['255.255.255.255']
256 không được     PASS  []


3000 chuỗi ngẫu nhiên: khớp brute force (thử mọi vị trí 3 dấu chấm)


# 8. Vì sao đúng? Độ phức tạp?

Mỗi IP hợp lệ ứng với đúng một cách chọn độ dài `(L1, L2, L3, L4)`, mỗi `Li ∈ {1, 2, 3}` và tổng bằng `len(s)`. Backtracking liệt kê mọi bộ như vậy (cắt tỉa chỉ bỏ các nhánh **không thể** đạt tổng đúng), và chỉ giữ những bộ có 4 phần hợp lệ.

Số nhánh tối đa `3^4 = 81` — **hằng số**. Mỗi kết quả dài tối đa 15 ký tự.

```text
Time:  O(1) (tối đa 81 nhánh, mỗi nhánh O(1) vì phần dài ≤ 3)
Space: O(1) (độ sâu đệ quy 4) — không tính output
```

Chuỗi dài hơn 12 ký tự bị loại ngay ở lời gọi đầu nhờ cắt tỉa.

# 9. Những lỗi dễ mắc

## Lỗi 1: Cho phép số 0 ở đầu

In [16]:
def wrong_leading_zero(s: str) -> List[str]:
    out = []
    for cuts in itertools.combinations(range(1, len(s)), 3):
        a, b, c = cuts
        parts = [s[:a], s[a:b], s[b:c], s[c:]]
        if all(len(p) <= 3 and int(p) <= 255 for p in parts):    # SAI: quên kiểm tra '0' đầu
            out.append(".".join(parts))
    return out

print("Sai :", sorted(wrong_leading_zero("10100")), "| Đúng:", sorted(sol.restoreIpAddresses("10100")))

Sai : ['1.0.1.00', '1.0.10.0', '1.01.0.0', '10.1.0.0'] | Đúng: ['1.0.10.0', '10.1.0.0']


## Lỗi 2: Không kiểm tra dùng hết chuỗi

Dừng khi đủ 4 phần mà không kiểm tra `start == len(s)` → chấp nhận IP chỉ dùng **tiền tố** của `s`.

In [17]:
def wrong_not_all_used(s: str) -> List[str]:
    result, parts = [], []
    def backtrack(start):
        if len(parts) == 4:
            result.append(".".join(parts))          # SAI: chưa chắc đã dùng hết s
            return
        for L in range(1, 4):
            p = s[start:start + L]
            if len(p) == L and valid_part(p):
                parts.append(p)
                backtrack(start + L)
                parts.pop()
    backtrack(0)
    return result

print("Sai :", wrong_not_all_used("11111")[:4], "... (", len(wrong_not_all_used("11111")), "kết quả ) | Đúng:", sol.restoreIpAddresses("11111"))

Sai : ['1.1.1.1', '1.1.1.11', '1.1.11.1', '1.11.1.1'] ... ( 5 kết quả ) | Đúng: ['1.1.1.11', '1.1.11.1', '1.11.1.1', '11.1.1.1']


## Lỗi 3: `int(p) <= 255` khi `p` rỗng

`int("")` gây `ValueError`. Phải chắc `p` không rỗng (kiểm tra độ dài, hoặc giới hạn `L <= remaining` như bản LeetCode).

In [18]:
try:
    int("")
except ValueError as e:
    print("Lỗi:", e)

Lỗi: invalid literal for int() with base 10: ''


## Lỗi 4: Kiểm tra `int(p) < 255`

`255` hợp lệ. Dùng `<= 255`.

# 10. Mẹo nhận diện pattern

**"Chia chuỗi thành các phần thỏa điều kiện"** → backtracking theo **vị trí cắt**: mỗi tầng chọn độ dài phần tiếp theo.

- Số phần cố định & nhỏ → có thể dùng vòng lặp lồng nhau.
- Có ràng buộc độ dài mỗi phần → cắt tỉa theo số ký tự còn lại.

Bài liên quan: **#131 Palindrome Partitioning** (mỗi phần là palindrome), **#842 Split Array into Fibonacci Sequence**, **#306 Additive Number**, **#468 Validate IP Address**, **#91 Decode Ways** (đếm thay vì liệt kê).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Liệt kê mọi IP hợp lệ từ `"1111"` và `"010010"`.

## Bài 2 — Điền chỗ trống

In [19]:
def exercise_2_restore_ip(s: str) -> List[str]:
    result, parts = [], []
    def backtrack(start):
        k = 4 - len(parts)
        remaining = len(s) - start
        # TODO: cắt tỉa; nếu k == 0 thì lưu; thử L = 1..3
        pass
    backtrack(0)
    return result


# assert sorted(exercise_2_restore_ip("25525511135")) == ["255.255.11.135", "255.255.111.35"]
# assert exercise_2_restore_ip("0000") == ["0.0.0.0"]
# assert exercise_2_restore_ip("123") == []

## Bài 3 — Biến thể: Validate IP Address (LeetCode #468, phần IPv4)

Kiểm tra một chuỗi (có dấu chấm sẵn) có phải IPv4 hợp lệ không.

```python
is_ipv4("172.16.254.1")   -> True
is_ipv4("172.16.254.01")  -> False
is_ipv4("256.256.256.256") -> False
is_ipv4("1.2.3")          -> False
is_ipv4("1.2.3.a")        -> False
```

In [20]:
def exercise_3_is_ipv4(ip: str) -> bool:
    # TODO
    pass


# assert exercise_3_is_ipv4("172.16.254.1") is True
# assert exercise_3_is_ipv4("172.16.254.01") is False
# assert exercise_3_is_ipv4("256.256.256.256") is False
# assert exercise_3_is_ipv4("1.2.3") is False
# assert exercise_3_is_ipv4("1.2.3.a") is False
# assert exercise_3_is_ipv4("1..2.3") is False

## Bài 4 — Biến thể: Palindrome Partitioning (LeetCode #131)

Chia `s` thành các phần sao cho **mỗi phần là palindrome**. Trả về mọi cách chia.

```python
partition("aab") -> [["a", "a", "b"], ["aa", "b"]]
```

In [21]:
def exercise_4_palindrome_partition(s: str) -> List[List[str]]:
    # TODO
    pass


# assert sorted(exercise_4_palindrome_partition("aab")) == [["a", "a", "b"], ["aa", "b"]]
# assert exercise_4_palindrome_partition("a") == [["a"]]
# assert len(exercise_4_palindrome_partition("aaaa")) == 8

## Bài 5 — Mini challenge: đếm số IP mà không liệt kê

Đếm số IP hợp lệ từ `s` bằng **DP** `dp[i][k]` = số cách chia `s[:i]` thành `k` phần hợp lệ.

In [22]:
def exercise_5_count_ips(s: str) -> int:
    # TODO
    pass


# assert exercise_5_count_ips("25525511135") == 2
# assert exercise_5_count_ips("101023") == 5
# assert exercise_5_count_ips("1" * 13) == 0

# 12. Gợi ý

**Bài 1**
- Mức 1: `"1111"` chỉ có `1.1.1.1`. `"010010"`: phần đầu phải là `"0"`.

**Bài 3**
- Mức 1: `parts = ip.split(".")`; cần đúng 4 phần.
- Mức 2: mỗi phần: không rỗng, `isdigit()`, `valid_part`.

**Bài 4**
- Mức 1: như backtracking ở mục 5, nhưng mọi độ dài `L` từ 1 tới hết chuỗi, điều kiện là `p == p[::-1]`.

**Bài 5**
- Mức 1: `dp[0][0] = 1`; `dp[i][k] = Σ_{L=1..3} dp[i-L][k-1]` nếu `s[i-L:i]` hợp lệ.
- Mức 2: đáp án `dp[n][4]`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
print(sol.restoreIpAddresses("1111"))
print(sol.restoreIpAddresses("010010"))

['1.1.1.1']
['0.10.0.10', '0.100.1.0']


## Đáp án Bài 2

In [24]:
def answer_2_restore_ip(s: str) -> List[str]:
    return restore_backtrack(s)

assert sorted(answer_2_restore_ip("25525511135")) == ["255.255.11.135", "255.255.111.35"]
assert answer_2_restore_ip("0000") == ["0.0.0.0"]
assert answer_2_restore_ip("123") == []
print("Bài 2 PASS (giống restore_backtrack ở mục 5)")

Bài 2 PASS (giống restore_backtrack ở mục 5)


## Đáp án Bài 3

In [25]:
def answer_3_is_ipv4(ip: str) -> bool:
    parts = ip.split(".")
    if len(parts) != 4:
        return False
    return all(p.isdigit() and valid_part(p) for p in parts)

assert answer_3_is_ipv4("172.16.254.1") is True
assert answer_3_is_ipv4("172.16.254.01") is False
assert answer_3_is_ipv4("256.256.256.256") is False
assert answer_3_is_ipv4("1.2.3") is False
assert answer_3_is_ipv4("1.2.3.a") is False
assert answer_3_is_ipv4("1..2.3") is False
for _ in range(500):
    s = "".join(random.choice("0125") for _ in range(random.randint(4, 12)))
    for ip in sol.restoreIpAddresses(s):
        assert answer_3_is_ipv4(ip)
print("Bài 3 PASS")

Bài 3 PASS


`"".isdigit()` là `False`, nên phần rỗng (`"1..2.3"`) bị loại trước khi gọi `valid_part`.

## Đáp án Bài 4

In [26]:
def answer_4_palindrome_partition(s: str) -> List[List[str]]:
    result, parts = [], []
    def backtrack(start):
        if start == len(s):
            result.append(parts[:])
            return
        for end in range(start + 1, len(s) + 1):
            p = s[start:end]
            if p == p[::-1]:
                parts.append(p)
                backtrack(end)
                parts.pop()
    backtrack(0)
    return result

assert sorted(answer_4_palindrome_partition("aab")) == [["a", "a", "b"], ["aa", "b"]]
assert answer_4_palindrome_partition("a") == [["a"]]
assert len(answer_4_palindrome_partition("aaaa")) == 8
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [27]:
def answer_5_count_ips(s: str) -> int:
    n = len(s)
    dp = [[0] * 5 for _ in range(n + 1)]
    dp[0][0] = 1
    for i in range(1, n + 1):
        for k in range(1, 5):
            for L in range(1, 4):
                if i - L >= 0 and valid_part(s[i - L:i]):
                    dp[i][k] += dp[i - L][k - 1]
    return dp[n][4]

assert answer_5_count_ips("25525511135") == 2
assert answer_5_count_ips("101023") == 5
assert answer_5_count_ips("1" * 13) == 0
for _ in range(2000):
    s = "".join(random.choice("0012259") for _ in range(random.randint(1, 14)))
    assert answer_5_count_ips(s) == len(sol.restoreIpAddresses(s))
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Mỗi tầng backtracking chọn độ dài `1..3` cho phần tiếp theo; phần hợp lệ khi `"0"` hoặc không bắt đầu bằng `0`, và `<= 255`. Cắt tỉa: còn `k` phần thì số ký tự còn lại phải trong `[k, 3k]`. Tối đa 81 nhánh.

---

## Checklist trước khi nộp

- [ ] Loại số 0 ở đầu (`"01"`, `"00"`)?
- [ ] `<= 255` (không phải `< 255`)?
- [ ] Đủ 4 phần **và** dùng hết chuỗi?
- [ ] Không gọi `int("")`?